In [8]:
import torch
import numpy as np
import scipy.io as sio
import torch.nn.functional as F
import os
import random
from sklearn.neighbors import kneighbors_graph
from torch.nn.functional import one_hot

def constructW(X, k, sigma):
    from sklearn.preprocessing import MinMaxScaler
    n = X.shape[1]

    W = (kneighbors_graph(X.T.cpu(), k+1, mode = 'distance', include_self = False, metric = 'euclidean')).toarray()
    W[W == 0] = np.inf

    W = np.exp((-(W**2))/(sigma**2))
    W = np.maximum(W, W.T)
    return torch.tensor(W).type(dtype)


def sim_dissim(y, n, p):
    l = np.fix(n*p).astype(np.uint8)
    numbers = random.sample(range(n), l)
    S = torch.zeros([n, n]).type(dtype)
    D = torch.zeros([n, n]).type(dtype)
    for i in numbers:
        for j in numbers:
            if y[i] == y[j]:
                S[i, j] = 1
            else:
                D[i, j] = 1
    DS = torch.diag(torch.sum(S, dim=0))
    return D, S, DS


def MakeLabel(data, per, c, n):
    arr = np.arange(n)
    np.random.shuffle(arr)
    data = data[arr, :]

    data = data[data[:, -1].sort()[1]]

    DATA = []
    nc = np.zeros(c).astype(int)
    for i in range(c):
        DATA.append(data[data[:, -1] == i])
        nc[i] = DATA[i].shape[0]
        l = np.round(nc[i] * per).astype(np.uint8)
        if i == 0:
            Labeled = DATA[0][:l, :]
            Unlabeled = DATA[0][l:, :]
        else:
            Labeled = torch.cat((Labeled, DATA[i][:l, :]), 0)
            Unlabeled = torch.cat((Unlabeled, DATA[i][l:, :]), 0)

    u = Unlabeled.shape[0]
    arr = np.arange(u)
    np.random.shuffle(arr)
    Unlabeled = Unlabeled[arr, :]

    data = torch.cat((Labeled, Unlabeled), 0)
    return data


def Alg(A, V, maxiter, num_en, S, DS, D, lam1, lam2):
    H = []
    [n, k, b] = V.shape
    for i in range(num_en):
        H.append(V[:, :, i])
    a = torch.rand(num_en, 1).type(dtype)
    a = a/torch.sum(a)
    h = torch.zeros(num_en, 1).type(dtype)

    eps = torch.tensor(0.00001).type(dtype)
    for it in range(maxiter):
        for kk in range(num_en):
            NUM = A @ H[kk] + lam2 * S @ H[kk]
            DEN = H[kk] @ H[kk].T @ H[kk] + lam2 * DS @ H[kk] + (lam1/2) * D @ H[kk]

            E = (NUM/torch.maximum(DEN, eps))
            H[kk] = H[kk] * (E**(1/4))
            Ht = torch.cdist(H[kk], H[kk], p=2.0)
            h[kk] = torch.norm(A-H[kk] @ H[kk].T)**2 + lam1 * torch.norm(D * (H[kk]@H[kk].T), 1) + lam2 * torch.norm(S * Ht, 1)
            
        sumH=torch.sum(1/h)
        a = (1/h)/sumH

    return H, a

In [9]:
## GPU or CPU
GPU = False
if GPU:
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    print("num GPUs", torch.cuda.device_count())
    dtype = torch.cuda.FloatTensor
else:
    dtype = torch.FloatTensor
    print("CPU")


X, y = torch.load('iris.pt')

n, d = X.shape

data0 = torch.cat((X, y), 1)
p = 0.1
c = torch.unique(y).shape[0]


data = MakeLabel(data0, p, c, n)
X = data[:, :-1]
y = data[:, -1]

K = 8
sigma = 100

A = constructW(X.T, K, sigma)

lam1 = 10
lam2 = 0.001

iters = 10
num_en = 20
maxiter = 500


D, S, DS = sim_dissim(y, n, p)

V = torch.rand(n, c, num_en, iters).type(dtype)
AA = []
HH = []
M = []

for i in range(num_en):
    AA.append(A)
    HH.append(V[:, :, i, 0])
    M.append(torch.zeros([n, c]))
ACC = np.zeros([iters, num_en])
NMI = np.zeros([iters, num_en])

for kk in range(iters):

    [H, a] = Alg(A, V[:, :, :, kk], maxiter, num_en, S, DS, D, lam1, lam2)

    for j in range(num_en):
        M[j] = F.one_hot(torch.argmax(H[j], dim=1), num_classes=c).float()
    sA = torch.zeros(n, n).type(dtype)
    for j in range(num_en):
        sA = sA + a[j] * (M[j] @ M[j].T)
    sA = sA/torch.max(sA)
    print("iteration => ",kk + 1)

CPU


/var/folders/4r/bcgfrsvj5_53sxd5lsmkdb_h0000gn/T/ipykernel_7815/2096732039.py:23: DeprecationWarning: numpy.fix is deprecated. Use numpy.trunc instead, which is faster and follows the Array API standard.
  l = np.fix(n*p).astype(np.uint8)


iteration =>  1
iteration =>  2
iteration =>  3
iteration =>  4
iteration =>  5
iteration =>  6
iteration =>  7
iteration =>  8
iteration =>  9
iteration =>  10


In [10]:
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import adjusted_rand_score

for j in range(num_en):
    pred = torch.argmax(H[j], dim=1).cpu().numpy()
    true = y.cpu().numpy()

    nmi = normalized_mutual_info_score(true, pred)
    ari = adjusted_rand_score(true, pred)

    print(f"Model {j+1}: NMI = {nmi:.4f}, ARI = {ari:.4f}")

Model 1: NMI = 0.8226, ARI = 0.8184
Model 2: NMI = 0.8226, ARI = 0.8184
Model 3: NMI = 0.6519, ARI = 0.4499
Model 4: NMI = 0.6519, ARI = 0.4499
Model 5: NMI = 0.7356, ARI = 0.7278
Model 6: NMI = 0.8226, ARI = 0.8184
Model 7: NMI = 0.8031, ARI = 0.7874
Model 8: NMI = 0.8204, ARI = 0.8495
Model 9: NMI = 0.6519, ARI = 0.4499
Model 10: NMI = 0.8226, ARI = 0.8184
Model 11: NMI = 0.8226, ARI = 0.8184
Model 12: NMI = 0.6050, ARI = 0.4346
Model 13: NMI = 0.8226, ARI = 0.8184
Model 14: NMI = 0.5217, ARI = 0.4192
Model 15: NMI = 0.6575, ARI = 0.6313
Model 16: NMI = 0.8226, ARI = 0.8184
Model 17: NMI = 0.7437, ARI = 0.7415
Model 18: NMI = 0.8226, ARI = 0.8184
Model 19: NMI = 0.7701, ARI = 0.7759
Model 20: NMI = 0.7725, ARI = 0.7818


In [11]:
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import accuracy_score

def clustering_accuracy(true, pred):
    cm = confusion_matrix(true, pred)

    row_ind, col_ind = linear_sum_assignment(-cm)

    return cm[row_ind, col_ind].sum() / len(true)

In [12]:
for j in range(num_en):
    pred = torch.argmax(H[j], dim=1).cpu().numpy()
    true = y.cpu().numpy()

    acc = clustering_accuracy(true, pred)
    nmi = normalized_mutual_info_score(true, pred)
    ari = adjusted_rand_score(true, pred)

    print(
        f"Model {j+1}: "
        f"ACC = {acc:.4f}, "
        f"NMI = {nmi:.4f}, "
        f"ARI = {ari:.4f}"
    )

Model 1: ACC = 0.9333, NMI = 0.8226, ARI = 0.8184
Model 2: ACC = 0.9333, NMI = 0.8226, ARI = 0.8184
Model 3: ACC = 0.5467, NMI = 0.6519, ARI = 0.4499
Model 4: ACC = 0.5467, NMI = 0.6519, ARI = 0.4499
Model 5: ACC = 0.8933, NMI = 0.7356, ARI = 0.7278
Model 6: ACC = 0.9333, NMI = 0.8226, ARI = 0.8184
Model 7: ACC = 0.9200, NMI = 0.8031, ARI = 0.7874
Model 8: ACC = 0.9467, NMI = 0.8204, ARI = 0.8495
Model 9: ACC = 0.5467, NMI = 0.6519, ARI = 0.4499
Model 10: ACC = 0.9333, NMI = 0.8226, ARI = 0.8184
Model 11: ACC = 0.9333, NMI = 0.8226, ARI = 0.8184
Model 12: ACC = 0.5467, NMI = 0.6050, ARI = 0.4346
Model 13: ACC = 0.9333, NMI = 0.8226, ARI = 0.8184
Model 14: ACC = 0.7000, NMI = 0.5217, ARI = 0.4192
Model 15: ACC = 0.8533, NMI = 0.6575, ARI = 0.6313
Model 16: ACC = 0.9333, NMI = 0.8226, ARI = 0.8184
Model 17: ACC = 0.9000, NMI = 0.7437, ARI = 0.7415
Model 18: ACC = 0.9333, NMI = 0.8226, ARI = 0.8184
Model 19: ACC = 0.9200, NMI = 0.7701, ARI = 0.7759
Model 20: ACC = 0.9200, NMI = 0.7725, AR